# Decision tree comparison

A short, controlled comparison of this project's decision tree and scikit-learn's `DecisionTreeClassifier` using the Breast Cancer Wisconsin dataset. The models receive the same data and the same maximum depth. We compare test accuracy, prediction agreement, and tree size.

## Method

- Dataset: 569 samples, 30 numeric features, and 2 classes.
- Split: stratified 80/20 train/test split with `random_state=42`.
- Criterion: Gini impurity for both models.
- Maximum depth: 5 for both models.

This is a fair comparison of one controlled experiment, not a claim that the implementations are equivalent. Scikit-learn is a production implementation with different split-search details and more input validation. The custom model's `fit` arguments are passed explicitly because they control the training run.

In [ ]:
from pathlib import Path
import random
import sys

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier as SklearnDecisionTreeClassifier

from decision_tree import DecisionTreeClassifier

RANDOM_STATE = 42
MAX_DEPTH = 5
dataset = load_breast_cancer()
X_train, X_test, y_train, y_test = train_test_split(
    dataset.data,
    dataset.target,
    test_size=0.20,
    stratify=dataset.target,
    random_state=RANDOM_STATE,
)
print(f"Dataset: {dataset.data.shape[0]} samples, {dataset.data.shape[1]} features")
print(f"Train/test sizes: {len(y_train)}/{len(y_test)}")

In [ ]:
random.seed(RANDOM_STATE)
custom = DecisionTreeClassifier()
custom.fit(X_train, y_train, max_depth=MAX_DEPTH)

sklearn_tree = SklearnDecisionTreeClassifier(
    criterion="gini",
    max_depth=MAX_DEPTH,
    random_state=RANDOM_STATE,
)
sklearn_tree.fit(X_train, y_train)

custom_pred = custom.predict(X_test).astype(int)
sklearn_pred = sklearn_tree.predict(X_test).astype(int)

comparison = pd.DataFrame(
    {
        "model": ["Custom", "scikit-learn"],
        "test_accuracy": [
            accuracy_score(y_test, custom_pred),
            accuracy_score(y_test, sklearn_pred),
        ],
        "tree_depth": [custom.root.depth(), sklearn_tree.get_depth()],
        "node_count": [custom.root.count_nodes(), sklearn_tree.tree_.node_count],
    }
)
print(comparison.round(4).to_string(index=False))
agreement = np.mean(custom_pred == sklearn_pred)
different = np.sum(custom_pred != sklearn_pred)
print(f"Prediction agreement: {agreement:.2%} ({len(y_test) - different}/{len(y_test)})")
print(f"Different predictions: {different}")

In [ ]:
matrices = [
    (confusion_matrix(y_test, custom_pred), "Custom"),
    (confusion_matrix(y_test, sklearn_pred), "scikit-learn"),
]
for matrix, name in matrices:
    print(f"{name} confusion matrix (rows=true, columns=predicted):")
    print(matrix)

print(
    "\nInterpretation: matching or similar accuracy on this split is useful evidence "
    "that the custom implementation behaves plausibly on this dataset. It does "
    "not establish equivalence, general superiority, or production readiness."
)